# Plots for symmnet

In [ ]:
import json
import os
import pickle  # noqa: F401 — kept for legacy compatibility if needed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
SAVEFIG = True

# Point this at the sweep directory produced by sweep.py
# SWEEP_DIR = Path("../../results/symm_net/sweep")
SWEEP_DIR = Path("/tmp/symmnet_test/sweep/")

FIG_DIR = Path("../../figs/symm_net")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# define the style etc.
mpl.style.use("../../mystyle.mpl")
plt.style.use("tableau-colorblind10")
plt.rcParams["text.usetex"] = True
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amssymb}"

In [ ]:
def load_runs(
    sweep_dir: Path, dataset: str, algo: str, metric_key: str
) -> pd.DataFrame:
    """Load a metric time series for all seeds of one dataset/algo combination.

    Args:
        sweep_dir: Root sweep directory (contains dataset sub-dirs).
        dataset: Dataset name (e.g. "cifar10").
        algo: Algorithm section name (e.g. "sal").
        metric_key: Key inside metrics["scalars"] (e.g. "accuracy/test").

    Returns:
        DataFrame with one column per seed (named seed_0, seed_1, …) and one
        row per epoch.
    """
    df = pd.DataFrame()
    runs_dir = sweep_dir / dataset / algo
    for seed_dir in sorted(runs_dir.glob("seed_*")):
        path = seed_dir / "metrics.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        df[seed_dir.name] = data["scalars"].get(metric_key, [])
    return df

In [ ]:
def add_stats(df: pd.DataFrame) -> None:
    """Add mean and std columns across all seed columns (in-place)."""
    seed_cols = [c for c in df.columns if c.startswith("seed_")]
    df["mean"] = df[seed_cols].mean(axis=1)
    df["std"] = df[seed_cols].std(axis=1)

In [ ]:
def calc_final(df: pd.DataFrame, n_last: int = 5) -> tuple[float, float]:
    seed_cols = [c for c in df.columns if c.startswith("seed_")]
    av_last = df[seed_cols].tail(n_last).mean()
    return av_last.mean(), av_last.std()


def calc_final_all(
    dfs: dict[str, pd.DataFrame], n_last: int = 5
) -> tuple[pd.DataFrame, pd.DataFrame]:
    means = pd.DataFrame()
    stds = pd.DataFrame()
    for key, df in dfs.items():
        mean, std = calc_final(df, n_last=n_last)
        means[key] = [mean]
        stds[key] = [std]
    return means, stds

In [ ]:
def plot_epochs(fig, ax, df, label):
    ax.plot(df["mean"], label=label)
    ax.fill_between(
        list(range(len(df["mean"]))),
        df["mean"] - df["std"],
        df["mean"] + df["std"],
        alpha=0.3,
    )

In [ ]:
def plot_bars(fig, ax, df_means, df_stds):

    # means, stds = calc_final_all(dfs)
    algos = ["BP", "SAL", "FA", "KP", "RDD"]
    means = [df_means[i] for i in algos]
    stds = [df_stds[i] for i in algos]
    num = len(means)
    x = np.arange(num)
    width = 0.8

    tab_colors = [f"C{i}" for i in range(7)]
    fontcolors = ["white", "black", "black", "white", "black"]

    bars = ax.bar(x, means, width, yerr=stds, capsize=4, color=tab_colors[:num])
    ax.set_xticks(x)
    ax.set_xticklabels("")

    TEXT_LIMIT = 75.0

    for i, (bar, fc) in enumerate(zip(bars, fontcolors)):
        height = bar.get_height()
        mean = means[i]
        std = stds[i]
        label = f"{mean:.1f}±{std:.1f}"
        ax.text(
            bar.get_x() + bar.get_width() / 2,  # x-coordinate: center of the bar
            (
                height - std - 1.0 if height > TEXT_LIMIT else height + std + 1.0
            ),  # y-coordinate: slightly above the bar top, adjust as needed
            label,
            ha="center",  # horizontally center the label
            va=(
                "top" if height > TEXT_LIMIT else "bottom"
            ),  # vertically align the label to the bottom
            rotation=90,  # rotate the label by 45 degrees
            fontsize=10,
            color=fc,
        )

In [ ]:
# Maps YAML section names to display labels used in the plots
ALGO_LABELS: dict[str, str] = {
    "bp": "BP",
    "fa": "FA",
    "bp_w_fa": "BP + FA",
    "akrout": "KP",
    "scfa": "SC FA",
    "sal": "SAL",
    "rdd": "RDD",
}

# Algorithms shown in the upper epoch-curves panel (CIFAR-10 detail)
cifar10_algos = ["bp", "sal", "fa", "akrout", "rdd"]
cifar10_labels = [ALGO_LABELS[a] for a in cifar10_algos]

In [ ]:
# Create the main figure
fig = plt.figure(figsize=(18 / 2.54, 11 / 2.54))

# Set up main GridSpec with 2 rows (for 2 subfigures)
gs = gridspec.GridSpec(
    2,
    1,
    height_ratios=[1, 1.3],
    figure=fig,
    top=0.85,
    bottom=0.03,
    right=0.95,
    hspace=0.5,
)

# Upper subfigure: 1 row, 4 columns
gs_upper = gridspec.GridSpecFromSubplotSpec(
    1, 5, subplot_spec=gs[0], width_ratios=[1.0, 0.3, 1, 1, 1], wspace=0.1
)
# axes_upper[0] bleibt wie gehabt
ax0 = fig.add_subplot(gs_upper[0, 0])

# axes_upper[1], [2], [3] bekommen gemeinsame y-Achse
ax1 = fig.add_subplot(gs_upper[0, 2])
ax2 = fig.add_subplot(gs_upper[0, 3], sharey=ax1)
ax3 = fig.add_subplot(gs_upper[0, 4], sharey=ax1)
axes_upper = [ax0, ax1, ax2, ax3]

# Lower subfigure: 1 row, 3 columns
gs_lower = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[1], wspace=0.07)
ax1 = fig.add_subplot(gs_lower[0, 0])
ax2 = fig.add_subplot(gs_lower[0, 1], sharey=ax1)
ax3 = fig.add_subplot(gs_lower[0, 2], sharey=ax1)
axes_lower = [ax1, ax2, ax3]
for ax in axes_lower:
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
for ax in axes_lower[1:]:
    ax.yaxis.set_visible(False)
    ax.spines["left"].set_visible(False)

# test accuracy:

In [ ]:
dfs = {
    algo: load_runs(SWEEP_DIR, "cifar10", algo, "accuracy/test")
    for algo in cifar10_algos
}
for df in dfs.values():
    add_stats(df)

In [ ]:
for df, label in zip(dfs.values(), cifar10_labels):
    plot_epochs(fig, axes_upper[0], df, label)

axes_upper[0].set_xlabel("epochs")
axes_upper[0].set_ylabel("validation accuracy [\%]")
axes_upper[0].set_title("CIFAR-10")

In [ ]:
fig.legend(
    *axes_upper[0].get_legend_handles_labels(),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),  # leicht unterhalb des oberen Rands
    ncol=5,
    frameon=True
)

# angles:

In [ ]:
dfs = {
    algo: load_runs(SWEEP_DIR, "cifar10", algo, "symm/angle/0")
    for algo in cifar10_algos
}
for df in dfs.values():
    add_stats(df)

for df, label in zip(dfs.values(), cifar10_labels):
    plot_epochs(fig, axes_upper[1], df, label)

axes_upper[1].set_xlabel("epochs")
axes_upper[1].set_ylabel(r"$\measuredangle (\mathbf W^T, \mathbf B)\ [\mathrm{deg}]$")
axes_upper[1].set_yticks([0, 45, 90])
axes_upper[1].set_title("FC 1")

In [ ]:
dfs = {
    algo: load_runs(SWEEP_DIR, "cifar10", algo, "symm/angle/1")
    for algo in cifar10_algos
}
for df in dfs.values():
    add_stats(df)

for df, label in zip(dfs.values(), cifar10_labels):
    plot_epochs(fig, axes_upper[2], df, label)

axes_upper[2].set_xlabel("epochs")
axes_upper[2].set_title("FC 2")
axes_upper[2].tick_params(axis="y", labelleft=False)

In [ ]:
dfs = {
    algo: load_runs(SWEEP_DIR, "cifar10", algo, "symm/angle/2")
    for algo in cifar10_algos
}
for df in dfs.values():
    add_stats(df)

for df, label in zip(dfs.values(), cifar10_labels):
    plot_epochs(fig, axes_upper[3], df, label)

axes_upper[3].set_xlabel("epochs")
axes_upper[3].set_title("FC 3")
axes_upper[3].tick_params(axis="y", labelleft=False)

# different data sets

## table with extra values in SM:

In [ ]:
datasets_local = ["cifar10", "fmnist", "svhn"]
datasets_labels = ["CIFAR-10", "FMNIST", "SVHN"]
# All 7 algorithms for the summary table / bar plots
all_algos = ["bp", "fa", "bp_w_fa", "akrout", "scfa", "sal", "rdd"]
all_labels = [ALGO_LABELS[a] for a in all_algos]

means_list = []
stds_list = []
for dataset in datasets_local:
    dfs = {
        algo: load_runs(SWEEP_DIR, dataset, algo, "accuracy/test") for algo in all_algos
    }
    mean, std = calc_final_all(dfs)
    mean.columns = all_labels
    std.columns = all_labels
    means_list.append(mean)
    stds_list.append(std)

means = pd.concat(means_list, ignore_index=True)
stds = pd.concat(stds_list, ignore_index=True)
means.index = datasets_labels
stds.index = datasets_labels

In [ ]:
latex_df = means.copy()
for col in means.columns:
    latex_df[col] = (
        means[col].round(1).astype(str) + "$\\pm$" + stds[col].round(1).astype(str)
    )

latex_df

In [ ]:
with open("test_err_table.tex", "w") as f:
    f.write(latex_df.to_latex(column_format="l" + "c" * len(latex_df.columns)))

## CIFAR 10

In [ ]:
plot_bars(fig, axes_lower[0], means.loc["CIFAR-10"], stds.loc["CIFAR-10"])

bp_fa = means.loc["CIFAR-10", "BP + FA"]
axes_lower[0].axhline(
    bp_fa, color="green", linestyle="--", label="theo. upper limit", zorder=-1
)

axes_lower[0].set_ylim(60, 100)
axes_lower[0].set_title("CIFAR-10", pad=0)
axes_lower[0].set_ylabel("test accuracy [\%]")
axes_lower[0].legend()

## FMNIST

In [ ]:
plot_bars(fig, axes_lower[1], means.loc["FMNIST"], stds.loc["FMNIST"])

bp_fa = means.loc["FMNIST", "BP + FA"]
axes_lower[1].axhline(bp_fa, color="green", linestyle="--", zorder=-1)

axes_lower[1].set_title("Fashion-MNIST", pad=0)
axes_lower[1].tick_params(axis="y", labelleft=False)

## SVHN

In [ ]:
plot_bars(fig, axes_lower[2], means.loc["SVHN"], stds.loc["SVHN"])

bp_fa = means.loc["SVHN", "BP + FA"]
axes_lower[2].axhline(bp_fa, color="green", linestyle="--", zorder=-1)

axes_lower[2].set_title("SVHN", pad=0)
axes_lower[2].tick_params(axis="y", labelleft=False)

In [ ]:
display(fig)

In [ ]:
if SAVEFIG:
    fig.savefig("../figs/salnet.png", dpi=300)
    fig.savefig("../figs/salnet.pdf")
    fig.savefig("../figs/salnet.svg")